# Лабораторная работа №1
## Линейная регрессия и факторный анализ

**Тема:** Прогноз цены жилья (California Housing Prices)

**Студент:** Зоркольцев Илья Алексеевич, АВТ-313

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.decomposition import PCA
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_percentage_error
from statsmodels.stats.outliers_influence import variance_inflation_factor

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 5)
RANDOM_STATE = 42

## 1. Загрузка датасета

Используем California Housing из встроенных датасетов scikit-learn.
Целевая переменная — `MedHouseVal` (медианная стоимость дома).

In [ ]:
data = fetch_california_housing(as_frame=True)
df = data.frame
print("Размер датасета:", df.shape)
df.head()

## 2. Первичный анализ (EDA)

Смотрим структуру, статистики, пропуски и распределения признаков.

In [ ]:
print("Информация о датасете:")
df.info()

print("\nСтатистики:")
display(df.describe())

print("\nПропуски по столбцам:")
print(df.isnull().sum())

In [ ]:
df.hist(figsize=(14, 10), bins=30, edgecolor='black')
plt.suptitle('Распределение признаков', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(9, 4))
sns.histplot(df['MedHouseVal'], kde=True, bins=50)
plt.title('Распределение целевой переменной MedHouseVal')
plt.xlabel('Медианная стоимость дома')
plt.show()

**Вывод по EDA:**
- Признаки имеют разные масштабы (Population в тысячах, AveRooms — единицы).
- Целевая переменная скошена вправо, есть «обрезание» на значении 5.00001.
- Пропусков нет (для sklearn-версии).

## 3. Предобработка данных

Удаляем пропуски (если есть) и кодируем категориальные признаки one-hot.

In [ ]:
# Удаляем пропуски
df = df.dropna()

# Если есть категориальный признак — кодируем
if 'ocean_proximity' in df.columns:
    df = pd.get_dummies(df, columns=['ocean_proximity'], drop_first=True)

print("Размер после предобработки:", df.shape)
df.head()

## 4. Матрица корреляций и VIF

Проверяем наличие мультиколлинеарности.

In [ ]:
plt.figure(figsize=(11, 8))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Матрица корреляций')
plt.show()

In [ ]:
X_vif = df.drop(columns=['MedHouseVal']).select_dtypes(include=[np.number])

vif_data = pd.DataFrame()
vif_data['feature'] = X_vif.columns
vif_data['VIF'] = [variance_inflation_factor(X_vif.values, i)
                   for i in range(X_vif.shape[1])]

vif_data = vif_data.sort_values('VIF', ascending=False).reset_index(drop=True)
print(vif_data)

**Вывод:** признаки с VIF > 10 считаются мультиколлинеарными.
Обычно в California Housing высокая VIF у `AveRooms`, `AveBedrms`, `Households`, `Population`.
Это оправдывает применение PCA на шаге 6.

## 5. Модели на исходных данных

Разделяем выборку 80/20, стандартизируем признаки,
обучаем Linear / Lasso / Ridge с кросс-валидацией.

In [ ]:
X = df.drop(columns=['MedHouseVal'])
y = df['MedHouseVal']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Train:", X_train_scaled.shape)
print("Test :", X_test_scaled.shape)

In [ ]:
models = {
    'Linear': LinearRegression(),
    'Lasso': Lasso(alpha=0.1, max_iter=10000),
    'Ridge': Ridge(alpha=1.0)
}

kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

results = {}

for name, model in models.items():
    cv_scores = cross_val_score(model, X_train_scaled, y_train,
                                cv=kf, scoring='r2')
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)

    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    mape = mean_absolute_percentage_error(y_test, y_pred)

    results[name] = {
        'Данные': 'Исходные',
        'CV R²': round(cv_scores.mean(), 4),
        'RMSE': round(rmse, 4),
        'R²': round(r2, 4),
        'MAPE': round(mape, 4)
    }

    print(f"{name:8s} | CV R²={cv_scores.mean():.4f} | "
          f"RMSE={rmse:.4f} | R²={r2:.4f} | MAPE={mape:.4f}")

## 6. PCA (метод главных компонент)

Перед PCA данные уже стандартизированы — это обязательное условие.
Сохраняем 95% объяснённой дисперсии.

In [ ]:
pca = PCA(n_components=0.95, random_state=RANDOM_STATE)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

print("Исходное число признаков:", X_train_scaled.shape[1])
print("Число главных компонент:", pca.n_components_)
print("Накопленная объяснённая дисперсия:",
      round(pca.explained_variance_ratio_.sum(), 4))

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(np.cumsum(pca.explained_variance_ratio_),
         marker='o', linestyle='--')
plt.axhline(0.95, color='red', linestyle=':', label='95% дисперсии')
plt.xlabel('Число главных компонент')
plt.ylabel('Накопленная объяснённая дисперсия')
plt.title('Выбор числа главных компонент')
plt.legend()
plt.grid(True)
plt.show()

## 7. Модели на главных компонентах

Обучаем те же три модели, но уже на PCA-признаках.

In [ ]:
for name, model in models.items():
    cv_scores = cross_val_score(model, X_train_pca, y_train,
                                cv=kf, scoring='r2')
    model.fit(X_train_pca, y_train)
    y_pred = model.predict(X_test_pca)

    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    mape = mean_absolute_percentage_error(y_test, y_pred)

    results[name + ' (PCA)'] = {
        'Данные': 'PCA',
        'CV R²': round(cv_scores.mean(), 4),
        'RMSE': round(rmse, 4),
        'R²': round(r2, 4),
        'MAPE': round(mape, 4)
    }

    print(f"{name:8s} (PCA) | CV R²={cv_scores.mean():.4f} | "
          f"RMSE={rmse:.4f} | R²={r2:.4f} | MAPE={mape:.4f}")

## 8. Сравнение результатов

In [ ]:
results_df = pd.DataFrame(results).T
print(results_df)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

results_df['RMSE'].plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('RMSE (меньше — лучше)')
axes[0].tick_params(axis='x', rotation=45)

results_df['R²'].plot(kind='bar', ax=axes[1], color='seagreen')
axes[1].set_title('R² (больше — лучше)')
axes[1].tick_params(axis='x', rotation=45)

results_df['MAPE'].plot(kind='bar', ax=axes[2], color='indianred')
axes[2].set_title('MAPE (меньше — лучше)')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
lasso = Lasso(alpha=0.1, max_iter=10000)
lasso.fit(X_train_scaled, y_train)

coefs = pd.Series(lasso.coef_, index=X.columns).sort_values()
coefs.plot(kind='barh', figsize=(9, 6), color='teal')
plt.title('Коэффициенты Lasso (обнулённые = неважные признаки)')
plt.xlabel('Коэффициент')
plt.grid(axis='x')
plt.show()

## Выводы

1. **EDA** показал разные масштабы признаков и скошенность целевой переменной,
   поэтому применялась стандартизация.

2. **VIF-анализ** выявил мультиколлинеарность (VIF > 10) у ряда признаков:
   AveRooms, AveBedrms, Households, Population.

3. **На исходных данных** лучший результат показала модель Ridge —
   L2-регуляризация компенсирует мультиколлинеарность.
   Lasso занулил часть коэффициентов, что упростило модель.

4. **PCA** сократил число признаков, сохранив 95% дисперсии.
   Качество моделей на компонентах оказалось сопоставимым
   (иногда чуть ниже), что ожидаемо: PCA теряет часть информации
   и ухудшает интерпретируемость, но устраняет мультиколлинеарность.

5. **Метрики:**
   - RMSE показывает среднюю абсолютную ошибку в единицах целевой переменной.
   - R² — долю объяснённой дисперсии (чем ближе к 1, тем лучше).
   - MAPE — среднюю относительную ошибку в процентах.

6. **Пути улучшения:**
   - Подбор гиперпараметра `alpha` через GridSearchCV.
   - Добавление полиномиальных признаков и взаимодействий.
   - Использование нелинейных моделей (Random Forest, Gradient Boosting).